In [1]:
import random
import os
from itertools import combinations
from collections import defaultdict

DATA_PATH = 'ml-100k/u.data'

print(f"Using dataset: {DATA_PATH}")

# Loading the data

print("Loading MovieLens 100k dataset...")
user_movies = defaultdict(set)

with open(DATA_PATH, 'r') as f:

    for line in f:
        parts = line.strip().split('\t')
        user_movies[int(parts[0])].add(int(parts[1]))

users  = sorted(user_movies.keys())
pairs  = list(combinations(users, 2))
print(f"  Users: {len(users)} | Movies: {len(set(m for ms in user_movies.values() for m in ms))} | Pairs: {len(pairs):,}")

# Exact Jaccard

def jaccard(a, b):

    i = len(a & b)
    u = len(a | b)
    return i / u if u > 0 else 0.0

print("\nComputing exact Jaccard for all pairs...")

exact_j = {}
for u, v in pairs:
    exact_j[(u, v)] = jaccard(user_movies[u], user_movies[v])

exact_above_06 = {p for p, j in exact_j.items() if j >= 0.6}
exact_above_08 = {p for p, j in exact_j.items() if j >= 0.8}

print(f"  Exact pairs with J >= 0.6 : {len(exact_above_06)}")
print(f"  Exact pairs with J >= 0.8 : {len(exact_above_08)}")

# MIN-HASH helpers

M = 1_000_003
P = 2**31 - 1

def make_hash_params(t, seed):
    random.seed(seed)
    return [(random.randint(1, P-1), random.randint(0, P-1)) for _ in range(t)]

def compute_signatures(params):
    """sig[user] = list of t min-hash values."""
    sigs = {}
    for user, movies in user_movies.items():
        sigs[user] = [min((a * mid + b) % M for mid in movies)
                      for a, b in params]
    return sigs

# LSH: candidate pairs via banding
# Two docs are candidates if ANY band has identical hash bucket

def lsh_candidates(sigs, r, b):
    """
    Split signature of length r*b into b bands of r rows.
    Return set of candidate pairs that share a bucket in >= 1 band.
    """
    candidates = set()
    for band_idx in range(b):
        buckets = defaultdict(list)
        for user in users:
            band_key = tuple(sigs[user][band_idx * r: (band_idx + 1) * r])
            buckets[band_key].append(user)
        for bucket_users in buckets.values():
            if len(bucket_users) > 1:
                for u, v in combinations(bucket_users, 2):
                    key = (min(u,v), max(u,v))
                    candidates.add(key)
    return candidates

# EXPERIMENT RUNNER

def run_experiment(t, r, b, threshold_set, threshold_label, num_runs=5):
    """
    Run LSH with given (t, r, b) over num_runs seeds.
    threshold_set: exact pairs above the similarity threshold.
    Returns avg FP, avg FN, and per-run details.
    """
    fps, fns = [], []
    for run in range(num_runs):

        seed   = run * 97 + t + r * 1000
        params = make_hash_params(t, seed)
        sigs   = compute_signatures(params)
        cands  = lsh_candidates(sigs, r, b)

        fp = cands - threshold_set          # candidate but exact J < threshold
        fn = threshold_set - cands          # exact J >= threshold but missed
        fps.append(len(fp))
        fns.append(len(fn))

    return fps, fns

# S-CURVE: theoretical probability of becoming a candidate
# f(s) = 1 - (1 - s^r)^b

def s_curve(s, r, b):
    return 1 - (1 - s**r)**b

# ALL EXPERIMENTS

configs = [
    # (t,   r,  b,  label)
    ( 50,   5, 10, "t=50,  r=5,  b=10"),
    (100,   5, 20, "t=100, r=5,  b=20"),
    (200,   5, 40, "t=200, r=5,  b=40"),
    (200,  10, 20, "t=200, r=10, b=20"),
]

NUM_RUNS = 5

for threshold, label, exact_set in [(0.6, "0.6", exact_above_06), (0.8, "0.8", exact_above_08),]:

    print("\n" + "=" * 70)
    print(f"  THRESHOLD tau = {label}   |   Exact pairs >= {label}: {len(exact_set)}")
    print("=" * 70)

    print(f"\n  {'Config':<22} {'s*(thresh)':>12} {'f(tau)':>10} "
          f"{'AvgFP':>8} {'AvgFN':>8}  {'FPs':>20}  {'FNs':>20}")
    print("  " + "-" * 115)

    for t, r, b, cfg_label in configs:

        fps, fns = run_experiment(t, r, b, exact_set, label, NUM_RUNS)
        avg_fp   = sum(fps) / NUM_RUNS
        avg_fn   = sum(fns) / NUM_RUNS
        s_star   = (1/b) ** (1/r)           # theoretical threshold
        f_tau    = s_curve(threshold, r, b)  # P(candidate) at tau

        print(f"  {cfg_label:<22} {s_star:>12.4f} {f_tau:>10.4f} "
              f"{avg_fp:>8.1f} {avg_fn:>8.1f}  {str(fps):>20}  {str(fns):>20}")

    # Detailed run-by-run breakdown
    print(f"\n  Detailed per-run results for tau = {label}:")
    for t, r, b, cfg_label in configs:

        fps, fns = run_experiment(t, r, b, exact_set, label, NUM_RUNS)
        print(f"\n  ── {cfg_label} ──")
        print(f"  {'Run':>5}  {'FP':>6}  {'FN':>6}")
        for i, (fp, fn) in enumerate(zip(fps, fns), 1):
            print(f"  {i:>5}  {fp:>6}  {fn:>6}")
        print(f"  {'Avg':>5}  {sum(fps)/NUM_RUNS:>6.1f}  {sum(fns)/NUM_RUNS:>6.1f}")


# Show exact ground pairs for reference

print("\n" + "=" * 70)
print("GROUND TRUTH: Exact pairs with Jaccard >= 0.6")
print("=" * 70)
print(f"  {'User A':>8} {'User B':>8} {'Exact J':>10}")
print("  " + "-" * 32)

for (u, v) in sorted(exact_above_06, key=lambda x: -exact_j[x]):
    print(f"  {u:>8} {v:>8} {exact_j[(u,v)]:>10.6f}")

print("\n" + "=" * 70)
print("GROUND TRUTH: Exact pairs with Jaccard >= 0.8")
print("=" * 70)
print(f"  {'User A':>8} {'User B':>8} {'Exact J':>10}")
print("  " + "-" * 32)

for (u, v) in sorted(exact_above_08, key=lambda x: -exact_j[x]):
    print(f"  {u:>8} {v:>8} {exact_j[(u,v)]:>10.6f}")


Using dataset: ml-100k/u.data
Loading MovieLens 100k dataset...
  Users: 943 | Movies: 1682 | Pairs: 444,153

Computing exact Jaccard for all pairs...
  Exact pairs with J >= 0.6 : 3
  Exact pairs with J >= 0.8 : 1

  THRESHOLD tau = 0.6   |   Exact pairs >= 0.6: 3

  Config                   s*(thresh)     f(tau)    AvgFP    AvgFN                   FPs                   FNs
  -------------------------------------------------------------------------------------------------------------------
  t=50,  r=5,  b=10            0.6310     0.5549    702.6      0.6  [1252, 891, 578, 401, 391]       [0, 1, 0, 0, 2]
  t=100, r=5,  b=20            0.5493     0.8019   1380.0      0.0  [1164, 1462, 1260, 1278, 1736]       [0, 0, 0, 0, 0]
  t=200, r=5,  b=40            0.4782     0.9608   2393.6      0.0  [2005, 2895, 2236, 1964, 2868]       [0, 0, 0, 0, 0]
  t=200, r=10, b=20            0.7411     0.1142      2.6      1.6       [3, 2, 4, 0, 4]       [1, 2, 2, 1, 2]

  Detailed per-run results for ta